[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cca-cce/osm-comm/blob/main/jnb/skod52/lecture_lab_1.ipynb)


---
title: "Web Scraping for Content Analysis"
subtitle: "SKOD52 · Module 3, Lecture-Lab 1 · companion notebook"
format:
  html:
    toc: true
    code-overflow: wrap
jupyter: python3
execute:
  warning: false
---

This notebook collects text from an organization's sustainability web pages
with **web scraping** instead of PDF captures. It follows the scraping slides
from the lecture:

1. **Your organization**: enter a website, check `robots.txt`, read the site map
2. **List of URLs**: up to 10 pages to collect
3. **Download HTML**: fetch each page and keep the raw HTML
4. **Parse and extract**: find the main text in the HTML with BeautifulSoup
5. **Text → pandas DataFrame**: one row per page (`df_web`)
6. **Compare with the PDF captures** from Lecture-Lab 2

Our example is the same organization as before: Equinor. Every output file is
written to the `text-data/web/` folder with a fixed name and is
**overwritten** on each run, so you can safely re-run the whole notebook from
the top.

## 0 · Setup

Everything we need is already installed on Google Colab: `requests`
downloads web pages, and `beautifulsoup4` (`bs4`) reads HTML. If `bs4` is
missing for some reason, the cell below installs it.

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

try:
    import bs4
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"], check=True)

print("Running on Colab:", IN_COLAB)

In [ ]:
import re, time
from datetime import date
from pathlib import Path
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 80)

### The data folder

As in Lecture-Lab 2, the notebook works with a folder called **`text-data`**
next to the notebook: `/content/text-data` on Colab. Scraped files go into the
subfolder `text-data/web/`, so they never overwrite the files from the PDF
workshop.

In [ ]:
DATA_DIR = Path("text-data")
WEB_DIR = DATA_DIR / "web"
WEB_DIR.mkdir(parents=True, exist_ok=True)
print("Data folder:", WEB_DIR.resolve())

## 1 · Your organization

Enter the web address of your organization's sustainability section in
`ORG_URL`. The notebook then tries to:

1. reach the website and download its **`robots.txt`**
2. find pages under `ORG_URL` that match `PATTERN`, in the **site map** and
   in the **links** on the `ORG_URL` page
3. pick at most `MAX_PAGES` of those pages that `robots.txt` allows

If this fails (the site blocks Colab, has no `robots.txt`, no matching pages,
…) the notebook falls back to the **Equinor example**, so all later cells
still run. The results are saved as three plain-text files in
`text-data/web/`: `robots.txt`, `sitemap.xml` and `urls.txt`.

We also act politely: a **User-Agent** header says who we are, and we wait
**`DELAY`** seconds between requests. Change the User-Agent text to describe
your own project.

In [ ]:
# ← EDIT: your organization's sustainability pages
ORG_URL = "https://www.equinor.com/sustainability"
PATTERN = r"sust"        # ← EDIT: regular expression the page URLs must match
MAX_PAGES = 10

# ← EDIT: say who you are (course, university) in your own words
HEADERS = {"User-Agent": "SKOD52 student project, Lund University (course web scraping exercise)"}
DELAY = 2   # seconds between requests

### Try the website, fall back to Equinor

`PATTERN` is a **regular expression**: `sust` matches *sustainability*,
*sustainable*, *sustainability-report* and so on, in any upper/lower case.
Use e.g. `sust|climate|esg|responsib` to cast a wider net. Short URLs (main
pages) are picked before long ones (news, sub-pages).

In [ ]:
EQUINOR_URL = "https://www.equinor.com/sustainability"
EQUINOR_PAGES = [   # best matches for the Lecture-Lab 2 PDF captures
    "https://www.equinor.com/sustainability/protecting-the-environment",
    "https://www.equinor.com/sustainability/health-safety-and-security",
    "https://www.equinor.com/sustainability/human-rights",
    "https://www.equinor.com/sustainability/governance-and-transparency",
    "https://www.equinor.com/sustainability/energy-transition-plan",
    "https://www.equinor.com/sustainability/just-transition",
    "https://www.equinor.com/sustainability/reporting",
    "https://www.equinor.com/sustainability/impact-assessments",
    "https://www.equinor.com/sustainability/energy-perspectives",
]


def fetch(url):
    """Download a URL; return the text, or None if it fails."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
    except requests.RequestException as e:
        print(f"   {url}: {type(e).__name__}")
        return None
    time.sleep(DELAY)
    return r.text if r.status_code == 200 else None


def sitemap_locs(xml):
    return [loc.get_text(strip=True) for loc in BeautifulSoup(xml, "xml").find_all("loc")]


def try_site(org_url):
    """Collect robots.txt, site map and page URLs for org_url; None if a step fails."""
    org_url = org_url.rstrip("/")
    parts = urlparse(org_url)
    site = f"{parts.scheme}://{parts.netloc}"

    org_html = fetch(org_url)
    print("✓" if org_html else "✗", "website reachable:", org_url)
    if not org_html:
        return None

    robots_txt = fetch(site + "/robots.txt")
    print("✓" if robots_txt is not None else "✗", "robots.txt downloaded")
    if robots_txt is None:
        return None
    robots = RobotFileParser()
    robots.parse(robots_txt.splitlines())

    sitemap_url = (robots.site_maps() or [site + "/sitemap.xml"])[0]
    sitemap_xml = fetch(sitemap_url)
    all_urls = sitemap_locs(sitemap_xml) if sitemap_xml else []
    if sitemap_xml and "<sitemapindex" in sitemap_xml:   # index: follow at most 5 site maps
        children, all_urls = all_urls[:5], []
        for child in children:
            all_urls += sitemap_locs(fetch(child) or "")
    print("✓" if all_urls else "–", f"site map: {len(all_urls)} URLs ({sitemap_url})")

    # second source: the links on the ORG_URL page itself
    links = [urljoin(org_url, a["href"]).split("#")[0]
             for a in BeautifulSoup(org_html, "lxml").find_all("a", href=True)]
    print("✓" if links else "–", f"links on the page: {len(links)}")

    matches = sorted({u.rstrip("/") for u in all_urls + links
                      if u.startswith(org_url) and re.search(PATTERN, u, re.IGNORECASE)
                      and robots.can_fetch(HEADERS["User-Agent"], u)},
                     key=lambda u: (u.count("/"), len(u)))
    print("✓" if matches else "✗", f"{len(matches)} allowed pages under {org_url} match {PATTERN!r}")
    if not matches:
        return None

    return {"site": site, "robots_txt": robots_txt, "sitemap_url": sitemap_url,
            "sitemap_xml": sitemap_xml, "matches": matches, "pages": matches[:MAX_PAGES]}


result = try_site(ORG_URL)
if result is None or ORG_URL.rstrip("/") == EQUINOR_URL:
    if result is None:
        print(f"\nWARNING: could not use {ORG_URL}, falling back to the Equinor example\n")
        result = try_site(EQUINOR_URL)
    result["pages"] = EQUINOR_PAGES

SITE = result["site"]
today = date.today().isoformat()
(WEB_DIR / "robots.txt").write_text(
    f"# downloaded from {SITE}/robots.txt on {today}\n" + result["robots_txt"], encoding="utf-8")
(WEB_DIR / "sitemap.xml").write_text(
    f"<!-- downloaded from {result['sitemap_url']} on {today} -->\n"
    + re.sub(r"^\s*<\?xml[^>]*\?>", "", result["sitemap_xml"]), encoding="utf-8")
(WEB_DIR / "urls.txt").write_text(
    f"# pages selected on {today}\n" + "\n".join(result["pages"]) + "\n", encoding="utf-8")

print(f"\nwebsite: {SITE}, {len(result['pages'])} pages saved to {WEB_DIR / 'urls.txt'}")

### Reading robots.txt

Here is the actual content of `robots.txt`. Each `User-agent:` line starts a
group of rules; `*` means all robots. `Disallow:` paths are off limits.
`robots.txt` can also point to a **site map**: a machine-readable list of the
pages that the site wants search engines to find.

In [ ]:
robots_txt = (WEB_DIR / "robots.txt").read_text(encoding="utf-8")
print(robots_txt)

robots = RobotFileParser()
robots.parse(robots_txt.splitlines())
print("Site maps listed in robots.txt:", robots.site_maps())

## 2 · List of URLs

::: {.callout-warning}
## Online content changes quickly

Organizations rename, rewrite and remove their web pages all the time. Some
of the Equinor pages that were captured as PDFs for Lecture-Lab 2 have since
been renamed or updated. So always **record the date** you collected a page
and **keep the raw HTML**. In your report, cite your own snapshot (or an
archived copy on <https://web.archive.org>) rather than only the live URL.
:::

### Pages found in the site map

A site map is either a list of pages (`<urlset>`) or an **index** of further
site maps (`<sitemapindex>`), for example one per language. These are the
first pages under `ORG_URL` that match `PATTERN`. Regular expressions can
surprise you: check for matches where the letters just happen to appear
inside another word.

In [ ]:
print(f"{len(result['matches'])} matching pages")
result["matches"][:15]

### Our list of pages

The pages to scrape are read from `urls.txt`. Each page gets a `text_id` from
the website name, e.g. `equinor01`, `equinor02`, … For the Equinor example we
use the pages that best match the nine PDF captures from Lecture-Lab 2. Four
of them have changed since the PDFs were made: the transition, just
transition and ESG reporting pages have new addresses or titles, and Energy
Perspectives now shows a newer edition.

To choose pages by hand, edit `urls.txt` (Colab: left sidebar → folder icon →
`text-data/web` → double-click the file) and re-run from this cell.

In [ ]:
def read_url_list(path):
    """One URL per line; empty lines and lines starting with # are skipped."""
    lines = path.read_text(encoding="utf-8").splitlines()
    return [l.strip() for l in lines if l.strip() and not l.strip().startswith("#")]


pages = read_url_list(WEB_DIR / "urls.txt")
name = urlparse(pages[0]).netloc.removeprefix("www.").split(".")[0]

df_urls = pd.DataFrame({
    "text_id": [f"{name}{i:02d}" for i in range(1, len(pages) + 1)],
    "url": pages,
})
df_urls["allowed"] = [robots.can_fetch(HEADERS["User-Agent"], u) for u in df_urls["url"]]
df_urls

## 3 · Download HTML

Now we download each page that `robots.txt` allows, wait `DELAY` seconds,
and save the raw HTML as `<text_id>.html`. A page that does not answer with
status code **200** (OK) gets a warning and is skipped.

In [ ]:
html_paths = []
for text_id, url, allowed in df_urls.itertuples(index=False):
    if not allowed:
        print(f"WARNING: robots.txt does not allow {url}, skipped")
        continue

    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        print(f"WARNING: {url} returned status {r.status_code}, skipped")
        continue

    html_path = WEB_DIR / f"{text_id}.html"
    html_path.write_text(r.text, encoding="utf-8")   # overwrite
    html_paths.append(html_path)
    print(f"{r.status_code}  {text_id}  {len(r.text) // 1024} kB")
    time.sleep(DELAY)

print(f"{len(html_paths)} HTML files in {WEB_DIR}")

## 4 · Parse and extract

An HTML page is a tree of **tags**: `<h1>` for the main heading, `<p>` for
paragraphs, `<li>` for list items, and so on. Besides the text we want, it
also contains menus (`<nav>`), a header and footer, and lots of JavaScript
(`<script>`). BeautifulSoup lets us find exactly the parts we want.

### Look at one page first

Always inspect the HTML before writing extraction code. Here we find
something surprising: the `<title>` tag is **empty** on these pages. The page
title is in the `og:title` meta tag (used by social media previews) and in
the `<h1>` heading instead.

In [ ]:
soup = BeautifulSoup(html_paths[0].read_text(encoding="utf-8"), "lxml")

og = soup.find("meta", property="og:title")
print("<title>:      ", repr(soup.title.get_text() if soup.title else None))
print("og:title:     ", og["content"] if og else None)
print("<h1>:         ", soup.h1.get_text(strip=True) if soup.h1 else None)
print("<article> tags:", len(soup.find_all("article")))

### All text vs. the main text

`soup.get_text()` returns **all** text on the page, including menus and the
footer. The main content of these pages is inside the `<article>` tag, so we
take only that, and remove the breadcrumb menu (`<nav>`) and any scripts
first.

In [ ]:
def page_title(soup):
    og = soup.find("meta", property="og:title")
    if og and og.get("content"):
        return " ".join(og["content"].split())
    return " ".join(soup.h1.get_text(" ").split()) if soup.h1 else ""


def main_text(soup):
    main = soup.find("article") or soup.find("main") or soup.body
    for tag in main.find_all(["nav", "script", "style", "header", "footer"]):
        tag.decompose()   # remove boilerplate
    blocks = [el.get_text(" ", strip=True)
              for el in main.find_all(["h1", "h2", "h3", "p", "li"])]
    return "\n".join(b for b in blocks if b)


all_text = BeautifulSoup(html_paths[0].read_text(encoding="utf-8"), "lxml").get_text(" ", strip=True)
article_text = main_text(soup)

print("words, whole page:", len(all_text.split()))
print("words, main text: ", len(article_text.split()))
print()
print(article_text[:600])

Some short boilerplate lines (for example links to PDF documents) may still
be left in the main text. As in Lecture-Lab 2, we keep a bit of noise on
purpose: real data is never perfectly clean.

## 5 · Text → pandas DataFrame

We apply the same extraction to every page, save each main text as
`<text_id>.txt`, and collect everything in a DataFrame with one row per page.
The `scraped_at` column records the collection date.

In [ ]:
rows = []
for html_path in html_paths:
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "lxml")
    text_raw = main_text(soup)
    txt_path = html_path.with_suffix(".txt")
    txt_path.write_text(text_raw, encoding="utf-8")   # overwrite

    rows.append({
        "text_id": html_path.stem,
        "url": df_urls.set_index("text_id").loc[html_path.stem, "url"],
        "title": page_title(soup),
        "scraped_at": date.today().isoformat(),
        "text_wordcount": len(text_raw.split()),
        "text_raw": re.sub(r"\s+", " ", text_raw),
    })

df_web = pd.DataFrame(rows, columns=["text_id", "url", "title", "scraped_at",
                                     "text_wordcount", "text_raw"])
df_web.to_csv(WEB_DIR / "df_web.tsv", sep="\t", index=False)   # overwrite
df_web

To read the table back later, for example in Lecture-Lab 3, use
`pd.read_csv("text-data/web/df_web.tsv", sep="\t", keep_default_na=False)`.

## 6 · Compare with the PDF captures

If the `.txt` files from Lecture-Lab 2 are in `text-data/`, we can compare
the length of each page as a PDF capture and as scraped HTML. Differences come
from what each method picks up (menus, image captions, PDF links) and from
pages that have changed since the PDFs were captured.

In [ ]:
pdf_counts = {}
for txt in sorted(DATA_DIR.glob("equinor0*.txt")):
    pdf_counts[txt.stem[:9]] = len(txt.read_text(encoding="utf-8").split())

df_compare = df_web[["text_id", "title", "text_wordcount"]].rename(
    columns={"text_wordcount": "words_web"})
df_compare["words_pdf"] = df_compare["text_id"].map(pdf_counts)

if df_compare["words_pdf"].notna().any():
    ax = df_compare.plot.barh(x="text_id", y=["words_pdf", "words_web"],
                              color=["#c4c4c0", "#f05a22"], figsize=(7, 4))
    ax.invert_yaxis()
    ax.set_xlabel("words")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No Lecture-Lab 2 .txt files found in text-data/, comparison skipped.")

df_compare

## Output files

All files below are in `text-data/web/` and are overwritten on every run.

In [ ]:
outputs = sorted(WEB_DIR.iterdir())
pd.DataFrame({"file": [p.name for p in outputs],
              "size_kb": [round(p.stat().st_size / 1024, 1) for p in outputs]})

## Download the results (on Colab)

Files in `/content` are deleted when the Colab session ends. This cell packs
the `text-data/web` folder (HTML, `.txt` and `.tsv` files) into
`text-data-web.zip` and downloads it to your computer. Locally, the cell does
nothing, since the files are already on your disk.

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files

    zip_path = shutil.make_archive("text-data-web", "zip", root_dir=".", base_dir="text-data/web")   # overwrite
    files.download(zip_path)